# 🧩 Module 1.6 — Task Decomposition Strategies

**Domain 1 · Agentic Architecture & Orchestration** (27% of the exam)
**Task 1.6 · Task Decomposition Strategies**
**Source:** [claudecertificationguide.com/learn/1-agentic-architecture/1-6-task-decomposition](https://claudecertificationguide.com/learn/1-agentic-architecture/1-6-task-decomposition)

Back to plain single-turn completions this time — no tools, no agentic loop.
This module is about *how you carve work into calls*, not about what any one
call can do.

### 🎯 What you'll build

A 10-file mini-codebase with deliberately planted bugs, reviewed two ways:
once in a single pass under a shared token budget, and once as ten
independent per-file passes plus a cross-file integration pass. Then you'll
measure — not just assert — the difference.

### ✅ What you'll walk away knowing

1. When to use a fixed sequential pipeline vs. dynamic adaptive decomposition
2. What attention dilution actually is, and why it's architectural, not a
   model-capability problem
3. Why a stronger model or a better prompt can't fix it, but multi-pass
   architecture does
4. How a per-item + cross-item pass structure catches both local and
   cross-cutting issues that a single pass misses

---

> **🔬 How this notebook reproduces attention dilution honestly:** rather
> than hoping a strong model happens to get worse on its own over 10 files
> (modern models are pretty good at this — it might not happen reliably),
> the single-pass baseline below is given a **deliberately tight, shared
> token budget** across all 10 files. That's a real, unavoidable mechanism —
> a shared budget *must* be divided somehow — and it's a fair proxy for the
> module's "attention gets divided across items" claim: measurable,
> reproducible, and not a strawman. The per-file passes get a **much larger
> combined budget** (their own dedicated allowance each), which is exactly
> what multi-pass architecture actually buys you.
>
> **💳 Cost:** 12 real, short, single-turn completions (1 single-pass + 10
> per-file + 1 integration) — no agentic loops, so each is fast. Cents total,
> maybe 20–40 seconds combined.

## 🔧 Setup

Same as Module 1.1.

```bash
pip install anthropic
```

```bash
# Windows (PowerShell)
$env:ANTHROPIC_API_KEY = "sk-ant-..."

# macOS / Linux (bash/zsh)
export ANTHROPIC_API_KEY="sk-ant-..."
```


In [ ]:
import os
from anthropic import Anthropic

if not os.environ.get("ANTHROPIC_API_KEY"):
    raise RuntimeError(
        "ANTHROPIC_API_KEY is not set. Set it in your shell, then restart the "
        "kernel and run this cell again -- see the Setup section above."
    )

client = Anthropic()
MODEL = "claude-sonnet-5"

print("Connected. Using model:", MODEL)


## 🔑 Key Concept: Fixed Sequential Pipelines (Prompt Chaining)

**Definition:** work broken into predetermined steps that always run in the
same order, each step's output feeding the next.

> "Step 1 runs, its output feeds into Step 2, Step 2's output feeds into
> Step 3, and so on. The sequence does not change based on intermediate
> results."

**Canonical example — code review pipeline:** per-file local analysis →
cross-file integration pass → unified report. (This is exactly what Tasks
2–4 below build.)

| | |
|---|---|
| **Best for** | Predictable, structured tasks: code reviews, document processing, data extraction, compliance checks |
| **Advantages** | Consistent (same input → same path), debuggable, easy to log at each stage |
| **Limitations** | Cannot adapt to unexpected findings; steps stay fixed regardless of what's discovered; wrong fit for open-ended investigation |


## 🔑 Key Concept: Dynamic Adaptive Decomposition

**Definition:** subtasks are generated from what's discovered at each step;
the plan evolves as the agent learns more.

**Canonical example — adding tests to a legacy codebase:** map the codebase
→ identify high-impact areas → build a prioritized test plan → start writing
tests → discover Module A depends on untested Module B → **reprioritize** to
test B first → keep adapting as new dependencies surface.

| | |
|---|---|
| **Best for** | Open-ended investigation with unknown scope: legacy exploration, security audits, research, debugging unfamiliar code |
| **Advantages** | Adapts to complexity as it's discovered; responds to surprises; more thorough for open-ended work |
| **Limitations** | Less predictable time/resource usage; harder to estimate completion or debug failures; variable behavior across similar-looking inputs |


## 🔑 Pattern Selection, at a Glance

| Task characteristics | Pattern | Why |
|---|---|---|
| Steps known in advance, structured input | Fixed pipeline | Consistency and reliability outweigh adaptability |
| Open-ended, unknown scope | Dynamic decomposition | Adaptability is essential when the problem isn't defined yet |
| Multi-file code review | Fixed pipeline | Per-file + cross-file analysis is predictable in advance |
| Legacy codebase exploration | Dynamic decomposition | Dependencies and issues only emerge during investigation |
| Document field extraction | Fixed pipeline | Fields and format are predetermined |
| Debugging an unfamiliar system | Dynamic decomposition | Root cause is unknown; the investigation has to adapt |


## 🔑 Key Concept: Attention Dilution — the Failure Mode This Module Is Really About

**Definition:** processing too many items in a single pass produces
**inconsistent** analysis depth — thorough for some items, superficial for
others — not uniformly-mediocre-for-everyone.

**Telltale symptoms:**
- Detailed feedback for the first few items; increasingly shallow feedback later
- The *same* pattern flagged as a problem in one item, silently approved in another
- Obvious issues missed in later items while minor ones get caught early

**Root cause, per the module:**

> "The model allocates attention across all items in the context. When there
> are too many items, attention per item decreases. Early items get
> disproportionate attention; later items get skimmed."

### Case study: the 14-file code review

A single-pass review of 14 files: files 1–5 get detailed, specific feedback;
6–9 get moderate feedback with some misses; 10–14 get superficial feedback
that misses real null-pointer bugs and a SQL-injection pattern outright. A
`forEach` loop gets flagged as inefficient in File 3 — and the *identical*
code in File 11 gets no comment at all.

**Diagnosis:** attention dilution — not model capability, not context-window
size.

### The structural fix: multi-pass architecture

1. **Per-item local passes** — analyze each file *individually*, its own
   pass, full attention (or full token budget) dedicated to just that one file.
2. **Cross-item integration pass** — one more pass, after all the local ones,
   specifically looking *across* items for things a per-item view can't see:
   inconsistent pattern usage, data-flow issues, cross-file dependencies.


## 🛠️ Build Exercise — Task 1: A 10-File Mini-Codebase, Bugs Included

**Objective:** at least 10 source files — the threshold the module's own
case study is built around (14, in their example) — with real, plantable
issues distributed across them.

**Why this matters:** attention dilution only becomes observable at this
kind of scale. Below, three specific patterns are placed deliberately:

- **Files `file_03.js` and `file_09.js` are byte-for-byte identical** — a
  `forEach`-based accumulator that a reviewer might reasonably flag as
  "prefer `reduce`". One early, one late — this is Task 6's consistency check.
- **Files `file_05.js`, `file_08.js`, and `file_10.js`** each have an
  unguarded, multi-level property chain (`a.b.c.d`) — a real null-pointer
  risk, skewed toward the *later* files, matching the case study's own
  "bugs missed in files 10–14" placement.
- **`file_07.js`** builds a SQL query via string interpolation of untrusted
  input — a classic, recognizable injection risk, placed in the middle.


In [ ]:
FILES = {
    "file_01.js": """function formatCurrency(amount) {
  return `$${amount.toFixed(2)}`;
}""",

    "file_02.js": """function slugify(text) {
  return text.toLowerCase().trim().replace(/\\s+/g, '-');
}""",

    "file_03.js": """function sumValues(items) {
  let total = 0;
  items.forEach(function (item) {
    total += item.value;
  });
  return total;
}""",

    "file_04.js": """function isValidEmail(email) {
  return /^[^\\s@]+@[^\\s@]+\\.[^\\s@]+$/.test(email);
}""",

    "file_05.js": """function getUserTheme(req) {
  return req.user.profile.settings.theme;
}""",

    "file_06.js": """function chunkArray(arr, size) {
  const chunks = [];
  for (let i = 0; i < arr.length; i += size) {
    chunks.push(arr.slice(i, i + size));
  }
  return chunks;
}""",

    "file_07.js": """function findUserById(db, userId) {
  const query = `SELECT * FROM users WHERE id = ${userId}`;
  return db.execute(query);
}""",

    "file_08.js": """function getShippingCity(order) {
  return order.customer.address.shipping.city;
}""",

    "file_09.js": """function sumValues(items) {
  let total = 0;
  items.forEach(function (item) {
    total += item.value;
  });
  return total;
}""",

    "file_10.js": """function getInvoiceContact(invoice) {
  return invoice.billing.contact.email.address;
}""",
}

print(f"Loaded {len(FILES)} files:", list(FILES.keys()))


## 🛠️ Build Exercise — Task 2: Single-Pass Baseline (Under a Shared Budget)

**Objective:** review all 10 files in **one** API call, all sharing a single
tight `max_tokens` budget, and see what that forces.

**Why this matters:** this reproduces the module's baseline symptom for
real: thorough early feedback, thinner (or missing) later feedback — not
because the model got worse, but because ten files are all drawing against
one shared allowance.

This cell makes 1 real API call.


In [ ]:
import re

SHARED_BUDGET_TOKENS = 700   # deliberately tight, shared across all 10 files

single_pass_prompt = (
    "Review the following JavaScript files for bugs, security issues, and "
    "code quality concerns. For EACH file, output a heading exactly in the "
    "form `## <filename>` followed by a short review comment for that file "
    "only. Cover every file.\n\n"
    + "\n\n".join(f"### {name}\n```javascript\n{content}\n```" for name, content in FILES.items())
)

single_pass_response = client.messages.create(
    model=MODEL,
    max_tokens=SHARED_BUDGET_TOKENS,
    messages=[{"role": "user", "content": single_pass_prompt}],
)

single_pass_text = "".join(b.text for b in single_pass_response.content if b.type == "text")

print("stop_reason:", single_pass_response.stop_reason)
if single_pass_response.stop_reason == "max_tokens":
    print("The response was cut off by the token budget before finishing --")
    print("an extreme, very literal form of the same problem: a shared budget")
    print("simply cannot stretch to cover everything once it runs out.")
print()
print(single_pass_text)


In [ ]:
def parse_per_file_sections(text: str, filenames) -> dict:
    """Split a `## filename` -- headed response into {filename: section_text}.
    Defensive on purpose: under a tight budget, some files may be cut off
    entirely and simply won't appear here -- that's a real result, not a
    parsing bug, and Task 5 reports on it rather than hiding it.
    """
    sections = {}
    pattern = "|".join(re.escape(name) for name in filenames)
    matches = list(re.finditer(rf"##\s+({pattern})\s*\n", text))
    for i, m in enumerate(matches):
        start = m.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        sections[m.group(1)] = text[start:end].strip()
    return sections


single_pass_sections = parse_per_file_sections(single_pass_text, FILES.keys())

print(f"Single-pass produced sections for {len(single_pass_sections)}/{len(FILES)} files.")
missing = [name for name in FILES if name not in single_pass_sections]
if missing:
    print("Missing entirely from the single-pass review:", missing)


## 🛠️ Build Exercise — Task 3: Per-File Local Analysis Passes

**Objective:** the same 10 files, each reviewed in its **own** API call with
its **own** generous token budget.

**Why this matters:** this is the structural fix, not a smarter prompt or a
bigger model — the exact same `MODEL` as Task 2, just called differently.
Ten independent 300-token budgets is a combined 3,000 tokens of allowance
across the review — over four times Task 2's shared 700 — and that
difference is entirely architectural.

This cell makes 10 real API calls (one per file).


In [ ]:
PER_FILE_BUDGET_TOKENS = 300  # generous, and DEDICATED to just one file each

per_file_reviews = {}

for name, content in FILES.items():
    prompt = (
        f"Review this JavaScript file for bugs, security issues, and code "
        f"quality concerns. Be specific.\n\nFile: {name}\n```javascript\n{content}\n```"
    )
    response = client.messages.create(
        model=MODEL,
        max_tokens=PER_FILE_BUDGET_TOKENS,
        messages=[{"role": "user", "content": prompt}],
    )
    review_text = "".join(b.text for b in response.content if b.type == "text")
    per_file_reviews[name] = review_text
    print(f"[{name}] ({len(review_text)} chars) {review_text[:80]}...")


## 🛠️ Build Exercise — Task 4: Cross-File Integration Pass

**Objective:** one more pass, taking all ten per-file reviews as input,
looking specifically for issues no single file's review could see on its own.

**Why this matters:** per-file passes catch *local* issues consistently —
they can't catch a pattern that's only a problem because it's inconsistent
*across* files. That needs its own dedicated pass. Skipping this step
(batching without integration) is Anti-Pattern 4, below.

This cell makes 1 real API call.


In [ ]:
integration_prompt = (
    "Here are independent code review summaries for 10 files in the same "
    "codebase. Identify CROSS-FILE issues only -- inconsistent pattern usage "
    "across files, data-flow concerns between modules, or the same code "
    "pattern being treated differently in different files. Do not repeat "
    "per-file issues that don't involve more than one file.\n\n"
    + "\n\n".join(f"## {name}\n{review}" for name, review in per_file_reviews.items())
)

integration_response = client.messages.create(
    model=MODEL,
    max_tokens=500,
    messages=[{"role": "user", "content": integration_prompt}],
)

integration_text = "".join(b.text for b in integration_response.content if b.type == "text")
print(integration_text)


## 🛠️ Build Exercise — Task 5: Compare, Quantitatively

**Objective:** measure the difference instead of just asserting it —
feedback length per file (a proxy for depth), and whether each approach's
review text even *mentions* the risk language a planted bug should trigger.

**Why this matters:** "multi-pass produces better reviews" is a claim
worth checking, not taking on faith — same as everywhere else in this repo.
Length isn't the same as quality, but a review that's been cut short
structurally cannot be as thorough as one that wasn't; it's a reasonable,
measurable proxy for the phenomenon, not a perfect one.


In [ ]:
print(f"{'file':<12} {'single-pass chars':>18} {'per-file chars':>16}")
for name in FILES:
    single_len = len(single_pass_sections.get(name, ""))
    multi_len = len(per_file_reviews.get(name, ""))
    flag = "  <- missing entirely" if name not in single_pass_sections else ""
    print(f"{name:<12} {single_len:>18} {multi_len:>16}{flag}")

print()
covered_single = len(single_pass_sections)
print(f"Single-pass covered {covered_single}/{len(FILES)} files at all under its shared budget.")
print(f"Per-file passes covered {len(per_file_reviews)}/{len(FILES)} files, each with its own budget.")


In [ ]:
RISK_LANGUAGE = {
    "file_05.js": ["null", "undefined", "optional chain", "?.", "typeerror", "throw"],
    "file_08.js": ["null", "undefined", "optional chain", "?.", "typeerror", "throw"],
    "file_10.js": ["null", "undefined", "optional chain", "?.", "typeerror", "throw"],
    "file_07.js": ["sql injection", "sanitiz", "parameteriz", "escape", "injection"],
}


def mentions_risk(text: str, keywords) -> bool:
    lowered = text.lower()
    return any(kw in lowered for kw in keywords)


print(f"{'file':<12} {'single-pass caught?':>20} {'per-file caught?':>18}")
for name, keywords in RISK_LANGUAGE.items():
    single_caught = mentions_risk(single_pass_sections.get(name, ""), keywords)
    multi_caught = mentions_risk(per_file_reviews.get(name, ""), keywords)
    print(f"{name:<12} {str(single_caught):>20} {str(multi_caught):>18}")

print()
print("('Caught' here means the review text used risk-relevant language for a")
print("planted bug -- a cheap, imperfect proxy for real understanding, but a")
print("concrete, checkable one rather than just trusting a summary.)")


## 🛠️ Build Exercise — Task 6: The `forEach` Consistency Artifact

**Objective:** `file_03.js` and `file_09.js` are identical. Did the
single-pass review treat them the same way? Did the per-file passes?

**Why this matters:** this is the case study's clearest, most concrete
symptom — "flagged in File 3, approved identically in File 11." If it shows
up here too, that's attention dilution made visible in your own output, not
just described in a guide.


In [ ]:
print("=== file_03.js and file_09.js (identical source) ===")
print(FILES["file_03.js"])
print()

print("=== Single-pass commentary ===")
print("file_03.js:", single_pass_sections.get("file_03.js", "(missing entirely)"))
print()
print("file_09.js:", single_pass_sections.get("file_09.js", "(missing entirely)"))
print()

print("=== Per-file commentary ===")
print("file_03.js:", per_file_reviews.get("file_03.js", "(missing)")[:300])
print()
print("file_09.js:", per_file_reviews.get("file_09.js", "(missing)")[:300])
print()

single_mentions_foreach_03 = "foreach" in single_pass_sections.get("file_03.js", "").lower()
single_mentions_foreach_09 = "foreach" in single_pass_sections.get("file_09.js", "").lower()
multi_mentions_foreach_03 = "foreach" in per_file_reviews.get("file_03.js", "").lower()
multi_mentions_foreach_09 = "foreach" in per_file_reviews.get("file_09.js", "").lower()

print(f"Single-pass mentioned 'forEach' for file_03: {single_mentions_foreach_03}, file_09: {single_mentions_foreach_09}")
print(f"Per-file mentioned 'forEach'    for file_03: {multi_mentions_foreach_03}, file_09: {multi_mentions_foreach_09}")

if single_mentions_foreach_03 != single_mentions_foreach_09:
    print()
    print("There it is: identical code, and the single-pass review treated the")
    print("two files differently -- exactly the inconsistency the case study describes.")
elif not single_pass_sections.get("file_09.js"):
    print()
    print("file_09.js didn't even get a section in the single-pass review -- an even")
    print("more extreme version of the same inconsistency: identical code, but only")
    print("one copy of it got any attention at all.")
else:
    print()
    print("Both got the same treatment this run -- real model output isn't scripted,")
    print("so this doesn't always reproduce on every single run. The per-file numbers")
    print("above still show the structural budget difference regardless.")


## ⚠️ Four Anti-Patterns to Avoid

| # | Anti-pattern | Why it fails | Fix |
|---|---|---|---|
| 1 | A stronger model or bigger context window | Attention dilution is architectural — it persists "regardless of model power or context size" | Multi-pass architecture (Tasks 3–4) |
| 2 | A better prompt asking for more thoroughness | Improves *average* quality; doesn't guarantee dedicated attention per item | Structural decomposition, not prompt tuning |
| 3 | A fixed pipeline for open-ended investigation | Can't adapt when the scope isn't known up front | Dynamic adaptive decomposition |
| 4 | Batching into smaller groups without a cross-file pass | Reduces within-batch dilution but misses issues *across* batches entirely | Always add a dedicated integration pass after per-item passes |

Each is written below as real code, then commented out.


In [ ]:
# ============================================================
# ❌ ANTI-PATTERN 1 -- "Just use a bigger/smarter model"
# ============================================================
# Commented out on purpose.
#
# response = client.messages.create(
#     model="claude-opus-5",       # <-- swapping in a stronger model
#     max_tokens=SHARED_BUDGET_TOKENS,
#     messages=[{"role": "user", "content": single_pass_prompt}],
# )
#
# Why it fails: the module is explicit that this is "an architectural "
# "problem, not a model capability problem." A stronger model sharing the
# same tight budget across 10 files still has to divide that budget
# somehow -- upgrading the model doesn't change the shape of the problem.


In [ ]:
# ============================================================
# ❌ ANTI-PATTERN 2 -- "Just write a better prompt"
# ============================================================
# Commented out on purpose.
#
# STRONGER_PROMPT_SUFFIX = (
#     "\n\nIMPORTANT: Be EQUALLY thorough for every single file, no matter "
#     "how many there are. Do not skimp on later files."
# )
#
# Why it fails: this asks the model to solve an allocation problem through
# willpower, in the same single call, against the same shared budget. Task 2
# already shows what a shared budget does regardless of instructions; a
# stronger prompt improves average quality, but "improves the average"
# is not the same guarantee as "every item gets dedicated attention."


In [ ]:
# ============================================================
# ❌ ANTI-PATTERN 3 -- A fixed pipeline for an open-ended task
# ============================================================
# Commented out on purpose.
#
# def debug_unfamiliar_system_antipattern(system):
#     step1 = check_logs(system)              # fixed step, always runs
#     step2 = check_recent_deploys(system)    # fixed step, always runs
#     step3 = check_database_health(system)   # fixed step, always runs
#     return summarize([step1, step2, step3])
#
# Why it fails: debugging an unfamiliar system is exactly the "root cause "
# "unknown; investigation must adapt" case from the decision table. If step 1
# reveals the real issue is a third-party API outage, this pipeline has no
# step for that -- it runs steps 2 and 3 anyway and never adapts.


In [ ]:
# ============================================================
# ❌ ANTI-PATTERN 4 -- Batching without an integration pass
# ============================================================
# Commented out on purpose.
#
# def review_in_batches_antipattern(files, batch_size=5):
#     names = list(files)
#     results = {}
#     for i in range(0, len(names), batch_size):
#         batch = {n: files[n] for n in names[i:i + batch_size]}
#         # ...review each batch, smaller shared budget per batch...
#         results.update(batch)  # no cross-BATCH pass afterward
#     return results
#
# Why it fails: smaller batches reduce dilution WITHIN each batch, but nothing
# here ever compares batch 1's files against batch 2's. A pattern-consistency
# issue (or a data-flow dependency) spanning file_03 in batch 1 and file_09 in
# batch 2 is invisible to this design -- Task 4's dedicated integration pass
# is the only place that comparison happens at all.


## 🎓 Practice Scenario (from the module)

> A code review agent processes 14 files and produces detailed feedback for
> the first 5 files but misses obvious bugs in files 10–14. It also flags a
> `forEach` loop as inefficient in one file while approving identical code
> in another. What is the root cause and most appropriate fix?
>
> - A. Reduce to 5 files per review, in sequential batches
> - B. Upgrade to a model with a larger context window
> - C. Add a stronger system prompt emphasizing thoroughness
> - D. Split into per-file local analysis passes plus a cross-file integration pass
>
> **Answer: D.** A creates smaller batches but still has no cross-batch
> comparison (Anti-Pattern 4). B confuses architecture with capability
> (Anti-Pattern 1). C asks nicely instead of restructuring (Anti-Pattern 2).
> Only D addresses the actual root cause — consistent depth per item, plus a
> dedicated pass for cross-item issues.


## 🏆 Key Takeaways for Exam Prep

1. **Match the pattern to the task** — fixed pipelines for predictable,
   structured work; dynamic decomposition for open-ended investigation with
   unknown scope.
2. **Attention dilution is architectural** — processing too many items in
   one pass guarantees inconsistent depth, regardless of model power,
   context size, or prompt quality.
3. **Multi-pass architecture is the structural fix** — per-item local passes
   for consistent depth, plus a cross-item integration pass for issues no
   single item's view can catch alone.
4. **The symptom is observable, not theoretical** — identical patterns
   treated differently across items, feedback that thins out or vanishes
   later in a batch, real bugs missed only in later items.
5. **The exam's favorite wrong answers** are a bigger model, a better
   prompt, and smaller batches with no integration pass. Recognize the
   shape of "asking nicely" dressed up as an architecture fix, and reject it.


---

## 🎉 Quick-Fire Recap — See If It Stuck

You reviewed the same 10 files two structurally different ways and measured
the difference yourself, instead of taking anyone's word for it. Try these
from memory first.

**1. In one line: what's the actual difference between a fixed pipeline and
dynamic decomposition?**
> 💡 A fixed pipeline's steps never change no matter what's found along the
> way. Dynamic decomposition regenerates its plan from what it just learned.

**2. Your teammate says "just use Opus instead of Sonnet" to fix a 14-file
review that gets shallow near the end. What do you tell them?**
> 💡 That's Anti-Pattern 1 — attention dilution is an architecture problem.
> A stronger model sharing the same single-pass budget across 14 files still
> has to divide it somehow.

**3. In your own Task 5 table, did every file get covered under the
single-pass budget, or did some go missing entirely?**
> 💡 Either result makes the point: partial coverage shows *some* files
> starved for space; total coverage with declining length per file shows the
> same allocation problem in a softer form.

**4. Why does adding a cross-file integration pass matter, if every file
already got its own dedicated per-file pass?**
> 💡 Because a per-file pass, by design, only ever sees one file. It has no
> way to notice that `file_03.js` and `file_09.js` are identical, or that two
> modules disagree about a shared data shape — that comparison only happens
> in a pass built to look across files.

**5. Someone proposes "smaller batches of 5 files instead of one batch of
10" as the fix. Does that solve the problem?**
> 💡 Partially — it reduces dilution *within* each batch, but Anti-Pattern 4
> still applies without a dedicated cross-batch pass: nothing ever compares
> batch 1 against batch 2.

**6. In your Task 6 check, did the single-pass review treat `file_03.js` and
`file_09.js` — literally identical code — the same way?**
> 💡 Whatever your run showed: that comparison, done for real on your own
> output, is the entire case study made concrete. A consistent per-file pass
> result on the same two files is the fix, demonstrated, not just described.

---

### 🚀 Nice work.

Six modules into Domain 1 — you've now covered single-agent loops,
multi-agent orchestration (hand-built and SDK-native), enforcement (hand-built
and hook-based), and how to structure the calls themselves so quality doesn't
quietly degrade as work scales up. Onward to **1.7 — Session State and
Resumption**, the last module in this domain.
